In [69]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [70]:
df = pd.read_csv('train.txt',sep = ';',header=None,names=['text','emotions'])

In [71]:
df

,text,emotions
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger
...,...,...
15995,i just had a very brief time in the beanbag an...,sadness
15996,i am now turning and i feel pathetic that i am...,sadness
15997,i feel strong and good overall,joy
15998,i feel like this was such a rude comment and i...,anger


In [72]:
df.isnull().sum()

,0
text,0
emotions,0


In [73]:
emotions = {}
unique_emotions = df['emotions'].unique()
i = 0
for emotion in unique_emotions:
  emotions[emotion] = i
  i+=1

df['emotions'] = df['emotions'].map(emotions)

In [74]:
df['text'] = df['text'].str.lower()

In [75]:
def removepunctuation(txt):
  return txt.translate(str.maketrans('','',string.punctuation))
  df['text'] = df['text'].apply(removepunctuation)

In [76]:
def removedigits(txt):
  return txt.translate(str.maketrans('','',string.digits))
  df['text'] = df['text'].apply(removedigits)

In [77]:
def removeemojis(txt):
  new = ""
  for i in txt:
        if i.isascii():
            new += i
  return new
  df['text'] = df['text'].apply(removeemojis)

In [78]:
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [79]:
def removestopwords(txt):
  cleaned = []
  words = txt.split()
  for word in words:
    if word not in stop_words:
     cleaned.append(word)

  return ' '.join(cleaned)

df['text'] = df['text'].apply(removestopwords)

In [80]:
df

,text,emotions
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1
...,...,...
15995,brief time beanbag said anna feel like beaten,0
15996,turning feel pathetic still waiting tables sub...,0
15997,feel strong good overall,5
15998,feel like rude comment im glad,1


In [81]:
from sklearn.model_selection import train_test_split
X = df['text']
y = df['emotions']

In [82]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [83]:
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,classification_report
bow = CountVectorizer()
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)


In [84]:
model_NB = MultinomialNB()
model_NB.fit(X_train_bow,y_train)
y_pred_nb = model_NB.predict(X_test_bow)
accuracy = accuracy_score(y_test, y_pred_nb)
print("Accuracy:", accuracy)
print(classification_report(y_test, y_pred_nb))

Accuracy: 0.768125
              precision    recall  f1-score   support

           0       0.75      0.95      0.84       946
           1       0.89      0.63      0.74       427
           2       0.93      0.27      0.41       296
           3       1.00      0.05      0.10       113
           4       0.85      0.57      0.68       397
           5       0.73      0.96      0.83      1021

    accuracy                           0.77      3200
   macro avg       0.86      0.57      0.60      3200
weighted avg       0.80      0.77      0.74      3200



In [88]:
model_NB_tfidf = MultinomialNB()
model_NB_tfidf.fit(X_train_tfidf,y_train)
y_pred_nb_tfidf = model_NB_tfidf.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred_nb_tfidf)
print("Accuracy:", accuracy)
print(classification_report(y_test, y_pred_nb_tfidf))

Accuracy: 0.6609375
              precision    recall  f1-score   support

           0       0.70      0.93      0.80       946
           1       0.93      0.29      0.44       427
           2       1.00      0.03      0.06       296
           3       1.00      0.01      0.02       113
           4       0.92      0.22      0.36       397
           5       0.60      0.99      0.74      1021

    accuracy                           0.66      3200
   macro avg       0.86      0.41      0.40      3200
weighted avg       0.76      0.66      0.58      3200



In [89]:
from sklearn.linear_model import LogisticRegression
lr_model = LogisticRegression()
lr_model.fit(X_train_bow,y_train)
y_pred_lr = lr_model.predict(X_test_bow)
accuracy = accuracy_score(y_test, y_pred_lr)
print("Accuracy:", accuracy)
print(classification_report(y_test, y_pred_lr))

Accuracy: 0.8896875
              precision    recall  f1-score   support

           0       0.93      0.93      0.93       946
           1       0.89      0.86      0.87       427
           2       0.85      0.76      0.80       296
           3       0.86      0.73      0.79       113
           4       0.86      0.84      0.85       397
           5       0.88      0.94      0.91      1021

    accuracy                           0.89      3200
   macro avg       0.88      0.84      0.86      3200
weighted avg       0.89      0.89      0.89      3200



In [90]:
lr_model_tfidf = LogisticRegression()
lr_model_tfidf.fit(X_train_tfidf,y_train)
y_pred_lr_tfidf = lr_model_tfidf.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred_lr_tfidf)
print("Accuracy:", accuracy)
print(classification_report(y_test, y_pred_lr_tfidf))

Accuracy: 0.8628125
              precision    recall  f1-score   support

           0       0.90      0.94      0.92       946
           1       0.90      0.81      0.86       427
           2       0.90      0.61      0.73       296
           3       0.88      0.47      0.61       113
           4       0.86      0.76      0.81       397
           5       0.81      0.96      0.88      1021

    accuracy                           0.86      3200
   macro avg       0.88      0.76      0.80      3200
weighted avg       0.87      0.86      0.86      3200



In [98]:
from sklearn import svm
svm_model = svm.LinearSVC()
svm_model.fit(X_train_bow,y_train)
y_pred_svm = svm_model.predict(X_test_bow)
accuracy = accuracy_score(y_test, y_pred_svm)
print("Accuracy:", accuracy)
print(classification_report(y_test, y_pred_svm))

Accuracy: 0.8903125
              precision    recall  f1-score   support

           0       0.93      0.92      0.93       946
           1       0.88      0.89      0.89       427
           2       0.82      0.79      0.80       296
           3       0.82      0.72      0.76       113
           4       0.85      0.86      0.86       397
           5       0.90      0.93      0.91      1021

    accuracy                           0.89      3200
   macro avg       0.87      0.85      0.86      3200
weighted avg       0.89      0.89      0.89      3200



In [100]:
svm_model_tfidf = svm.LinearSVC()
svm_model_tfidf.fit(X_train_tfidf,y_train)
y_pred_svm_tfidf = svm_model_tfidf.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred_svm_tfidf)
print("Accuracy:", accuracy)
print(classification_report(y_test, y_pred_svm_tfidf))

Accuracy: 0.891875
              precision    recall  f1-score   support

           0       0.93      0.93      0.93       946
           1       0.89      0.88      0.88       427
           2       0.83      0.74      0.78       296
           3       0.88      0.70      0.78       113
           4       0.87      0.85      0.86       397
           5       0.89      0.94      0.91      1021

    accuracy                           0.89      3200
   macro avg       0.88      0.84      0.86      3200
weighted avg       0.89      0.89      0.89      3200



In [101]:
from sklearn.linear_model import SGDClassifier
sgd_model = SGDClassifier()
sgd_model.fit(X_train_bow,y_train)
y_pred_sgd = sgd_model.predict(X_test_bow)
accuracy = accuracy_score(y_test, y_pred_sgd)
print("Accuracy:", accuracy)
print(classification_report(y_test, y_pred_sgd))

Accuracy: 0.8965625
              precision    recall  f1-score   support

           0       0.93      0.93      0.93       946
           1       0.90      0.89      0.89       427
           2       0.83      0.76      0.79       296
           3       0.89      0.71      0.79       113
           4       0.84      0.89      0.86       397
           5       0.91      0.93      0.92      1021

    accuracy                           0.90      3200
   macro avg       0.88      0.85      0.86      3200
weighted avg       0.90      0.90      0.90      3200



In [103]:
sgd_model_tfidf = SGDClassifier(loss='hinge')
sgd_model_tfidf.fit(X_train_tfidf,y_train)
y_pred_sgd_tfidf = sgd_model_tfidf.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred_sgd_tfidf)
print("Accuracy:", accuracy)
print(classification_report(y_test, y_pred_sgd_tfidf))

Accuracy: 0.893125
              precision    recall  f1-score   support

           0       0.92      0.94      0.93       946
           1       0.90      0.89      0.89       427
           2       0.85      0.73      0.78       296
           3       0.86      0.69      0.76       113
           4       0.87      0.85      0.86       397
           5       0.89      0.94      0.92      1021

    accuracy                           0.89      3200
   macro avg       0.88      0.84      0.86      3200
weighted avg       0.89      0.89      0.89      3200



In [ ]:
results = []

# Multinomial Naive Bayes (BOW)
results.append({
    'Model': 'Multinomial Naive Bayes',
    'Feature': 'BOW',
    'Accuracy': accuracy_score(y_test, y_pred_nb),
    'Precision': precision_score(y_test, y_pred_nb, average='weighted'),
    'Recall': recall_score(y_test, y_pred_nb, average='weighted'),
    'F1-Score': f1_score(y_test, y_pred_nb, average='weighted')
})

# Multinomial Naive Bayes (TF-IDF)
results.append({
    'Model': 'Multinomial Naive Bayes',
    'Feature': 'TF-IDF',
    'Accuracy': accuracy_score(y_test, y_pred_nb_tfidf),
    'Precision': precision_score(y_test, y_pred_nb_tfidf, average='weighted'),
    'Recall': recall_score(y_test, y_pred_nb_tfidf, average='weighted'),
    'F1-Score': f1_score(y_test, y_pred_nb_tfidf, average='weighted')
})

# Logistic Regression (BOW)
results.append({
    'Model': 'Logistic Regression',
    'Feature': 'BOW',
    'Accuracy': accuracy_score(y_test, y_pred_lr),
    'Precision': precision_score(y_test, y_pred_lr, average='weighted'),
    'Recall': recall_score(y_test, y_pred_lr, average='weighted'),
    'F1-Score': f1_score(y_test, y_pred_lr, average='weighted')
})

# Logistic Regression (TF-IDF)
results.append({
    'Model': 'Logistic Regression',
    'Feature': 'TF-IDF',
    'Accuracy': accuracy_score(y_test, y_pred_lr_tfidf),
    'Precision': precision_score(y_test, y_pred_lr_tfidf, average='weighted'),
    'Recall': recall_score(y_test, y_pred_lr_tfidf, average='weighted'),
    'F1-Score': f1_score(y_test, y_pred_lr_tfidf, average='weighted')
})

# SVM (BOW)
results.append({
    'Model': 'SVM',
    'Feature': 'BOW',
    'Accuracy': accuracy_score(y_test, y_pred_svm),
    'Precision': precision_score(y_test, y_pred_svm, average='weighted'),
    'Recall': recall_score(y_test, y_pred_svm, average='weighted'),
    'F1-Score': f1_score(y_test, y_pred_svm, average='weighted')
})

# SVM (TF-IDF)
results.append({
    'Model': 'SVM',
    'Feature': 'TF-IDF',
    'Accuracy': accuracy_score(y_test, y_pred_svm_tfidf),
    'Precision': precision_score(y_test, y_pred_svm_tfidf, average='weighted'),
    'Recall': recall_score(y_test, y_pred_svm_tfidf, average='weighted'),
    'F1-Score': f1_score(y_test, y_pred_svm_tfidf, average='weighted')
})

# SGD Classifier (BOW)
results.append({
    'Model': 'SGD Classifier',
    'Feature': 'BOW',
    'Accuracy': accuracy_score(y_test, y_pred_sgd),
    'Precision': precision_score(y_test, y_pred_sgd, average='weighted'),
    'Recall': recall_score(y_test, y_pred_sgd, average='weighted'),
    'F1-Score': f1_score(y_test, y_pred_sgd, average='weighted')
})

# SGD Classifier (TF-IDF)
results.append({
    'Model': 'SGD Classifier',
    'Feature': 'TF-IDF',
    'Accuracy': accuracy_score(y_test, y_pred_sgd_tfidf),
    'Precision': precision_score(y_test, y_pred_sgd_tfidf, average='weighted'),
    'Recall': recall_score(y_test, y_pred_sgd_tfidf, average='weighted'),
    'F1-Score': f1_score(y_test, y_pred_sgd_tfidf, average='weighted')
})

results_df = pd.DataFrame(results)

# Plotting
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

for metric in metrics:
    plt.figure(figsize=(12, 6))
    sns.barplot(x='Model', y=metric, hue='Feature', data=results_df, palette='viridis')
    plt.title(f'{metric} Comparison Across Models and Features')
    plt.ylim(0, 1) # Metrics are between 0 and 1
    plt.ylabel(metric)
    plt.xlabel('Model')
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Feature Type')
    plt.tight_layout()
    plt.show()

    